# GPAT

Gridded Plume Analysis Tool (GPAT) modelling framework. This simulates flight trajectories, estimates fuel burn and emissions, models dispersion effects, and aggregates plume data to a common Eulerian grid for further photochemical and microphysical processing.

In [1]:
import pandas as pd

from pycontrails.models.gpat.gpat import (
    GPAT,
    FlParams,
    PlumeParams,
    SimParams,
)
from pycontrails.models.gpat.pp_gpat import GPATPostProcessor

In [2]:
# flight trajectory parameters
fl_params = {
    "t0_fl": pd.to_datetime("2022-01-20 13:00:00"),  # flight start time
    "rt_fl": pd.Timedelta(minutes=60),  # flight run time
    "ts_fl": pd.Timedelta(minutes=2),  # flight time step
    "ac_type": "A320",  # aircraft type
    "fl0_speed": 100.0,  # m/s
    "fl0_heading": 45.0,  # deg
    "fl0_coords0": (0.1, 0.1, 10500),  # lat, lon, alt [deg, deg, m]
    "sep_dist": (5000, 2000, 0),  # dx, dy, dz [m]
    "n_ac": 1,  # number of aircraft
}

In [3]:
# plume dispersion parameters
plume_params = {
    "dt_integration": pd.Timedelta(minutes=2),  # integration time step
    "max_age": pd.Timedelta(hours=2),  # maximum age of the plume
    "depth": 50.0,  # initial plume depth, [m]
    "width": 50.0,  # initial plume width, [m]
    "verbose_outputs": False,  # print verbose outputs
    "hres_pl": 0.05,  # horizontal resolution of the plume [deg]
    "vres_pl": 500,  # vertical resolution of the plume [m]
}

In [4]:
# chemistry sim parameters
sim_params = {
    "t0_sim": pd.to_datetime("2022-01-20 12:00:00"),  # chemistry start time
    "rt_sim": pd.Timedelta(hours=12),  # chemistry runtime
    "ts_sim": pd.Timedelta(seconds=20),  # chemistry time step
    "lat_bounds": (0.0, 1.0),  # lat bounds [deg]
    "lon_bounds": (0.0, 1.0),  # lon bounds [deg]
    "alt_bounds": (10000, 11000),  # alt bounds [m]
    "hres_sim": 0.05,  # horizontal resolution [deg]
    "vres_sim": 500,  # vertical resolution [m]
    "eastward_wind": 0.0,  # m/s
    "northward_wind": 0.0,  # m/s
    "lagrangian_tendency_of_air_pressure": 0.0,  # m/s
    "species_in": ("NO", "NO2", "CO", "HCHO", "CH3CHO", "C2H4", "C3H6", "C2H2", "BENZENE"),
    "species_out": (
        "O3",
        "NO2",
        "NO",
        "NO3",
        "HNO3",
        "PAN",
        "HONO",
        "HO2",
        "OH",
        "H2O2",
        "CO",
        "HCHO",
        "CH4",
        "CH3O2",
    ),
    "run_path": "/home/ktait98/GPAT2025/pycontrails_kt/pycontrails/models/gpat/",
    "data_path": "/home/ktait98/GPAT2025/pycontrails_kt/pycontrails/models/gpat/data/",  # "/projects/Impact_of_aviation_on_climate
    "job_id": "GPAT_test_1_ac",
    "run_gpat": True,
}

In [5]:
fl_params = FlParams(**fl_params)
plume_params = PlumeParams(**plume_params)
sim_params = SimParams(**sim_params)

# updated_args = parse_args()

# update_fl_params_from_args(fl_params, updated_args)
# print("FlParams:", asdict(fl_params))

# update_plume_params_from_args(plume_params, updated_args)
# print("PlumeParams:", asdict(plume_params))

# update_sim_params_from_args(sim_params, updated_args)

# print("SimParams:", asdict(sim_params))

gpat = GPAT(fl_params, plume_params, sim_params)

if gpat.sim_params.run_gpat:
    gpat.eval()
else:
    print("GPAT simulation is not run.")
    print(f"Job ID is : {gpat.sim_params.job_id}")

flight 0 done


/home/ktait98/GPAT2025/pycontrails_kt/pycontrails/models/gpat/gpat.py:750: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  fl[i].dataframe[column] = fl[i].dataframe[column].fillna(method="ffill")
/home/ktait98/GPAT2025/pycontrails_kt/pycontrails/models/gpat/gpat.py:806: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  fl[i][column] = fl[i][column].fillna(method="ffill")


KeyError: "['sigma_zz'] not in index"

In [ ]:
outputs_dir = f"{sim_params.data_path}/outputs/"
criteria = {}

pp_gpat = GPATPostProcessor(outputs_dir, criteria)

In [ ]:
# Create dicts to hold all necessary data (but no more)
fl_df_dict = {}
pl_df_dict = {}
chem_ds_dict = {}

pp_gpat.filtered_df

In [ ]:
for job_id in pp_gpat.job_ids:
    # fl_df_dict[job_id] = pp_gpat.load_fl_df(job_id)
    # pl_df_dict[job_id] = pp_gpat.load_pl_df(job_id)
    chem_ds_dict[job_id] = pp_gpat.load_chem_ds(job_id)

In [ ]:
chem_ds = chem_ds_dict["GPAT_test"]

In [ ]:
chem_ds.Y.sel(species_out="NO2").isel(latitude=10, longitude=10).plot()